### 📚 Building a GPT Dataset and DataLoader (Step-by-Step Explanation)

This code prepares **training data for a GPT-style language model**.

A GPT model learns by predicting the **next token in a sequence**.

Example:

```
Input :  "The cat sat on"
Target:  "cat sat on the"
```

Each token in the input tries to **predict the next token**.

---

##### 🧱 Step 1 — Import Required Libraries

```python
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
```

### 🔹 `tiktoken`

Tokenizer used by GPT models.

Converts text → tokens.

Example

```
"The cat sat"
↓
[464, 3797, 3332]
```

---

### 🔹 `torch`

Main PyTorch library used for:

* tensor operations
* neural networks
* training

---

### 🔹 `torch.nn`

Contains neural network layers like:

```
nn.Linear
nn.Embedding
nn.LayerNorm
```

---

### 🔹 `Dataset` and `DataLoader`

These are PyTorch utilities used to handle training data.

| Class        | Purpose               |
| ------------ | --------------------- |
| `Dataset`    | stores the dataset    |
| `DataLoader` | loads data in batches |

---

##### 🧱 Step 2 — Creating the Dataset Class

```python
class GPTDatasetV1(Dataset):
```

This class creates a **dataset suitable for GPT training**.

GPT training requires:

```
Input tokens
Target tokens (next tokens)
```

---

# 🧠 Step 3 — Dataset Initialization

```python
def __init__(self, txt, tokenizer, max_length, stride):
```

### Parameters

| Parameter    | Meaning                      |
| ------------ | ---------------------------- |
| `txt`        | training text                |
| `tokenizer`  | converts text → tokens       |
| `max_length` | sequence length              |
| `stride`     | step size for sliding window |

---

# 🧠 Step 4 — Storage Lists

```python
self.input_ids = []
self.target_ids = []
```

Two lists are created:

| List         | Purpose                |
| ------------ | ---------------------- |
| `input_ids`  | input tokens           |
| `target_ids` | expected output tokens |

---

###### 🧠 Step 5 — Tokenizing the Text

```python
token_ids = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})
```

The entire text is converted into tokens.

Example:

```
Text:
"Hello world"

Tokens:
[15496, 995]
```

So the entire book becomes:

```
[15496, 995, 23, 1874, 29, ...]
```

---

#### 🧠 Step 6 — Sliding Window Chunking

```python
for i in range(0, len(token_ids) - max_length, stride):
```

This creates **overlapping chunks**.

Why?

Because GPT needs **fixed length sequences**.

---

### Example

Suppose tokens are:

```
[10,11,12,13,14,15,16,17]
```

If

```
max_length = 4
stride = 4
```

Chunks become:

```
[10,11,12,13]
[14,15,16,17]
```

---

If

```
stride = 2
```

Chunks become overlapping:

```
[10,11,12,13]
[12,13,14,15]
[14,15,16,17]
```

Overlapping improves learning.

---

#### 🧠 Step 7 — Creating Input Sequence

```python
input_chunk = token_ids[i:i + max_length]
```

Example

```
[10,11,12,13]
```

This is the **model input**.

---

#### 🧠 Step 8 — Creating Target Sequence

```python
target_chunk = token_ids[i + 1: i + max_length + 1]
```

Example

```
[11,12,13,14]
```

This is the **next token prediction target**.

---

### Visual Representation

```
Input  : [10,11,12,13]
Target : [11,12,13,14]
```

So the model learns:

```
10 → 11
11 → 12
12 → 13
13 → 14
```

---

#### 🧠 Step 9 — Converting to PyTorch Tensor

```python
self.input_ids.append(torch.tensor(input_chunk))
self.target_ids.append(torch.tensor(target_chunk))
```

Why?

Neural networks work with **tensors**, not Python lists.

Example tensor

```
tensor([10,11,12,13])
```

---

#### 🧱 Step 10 — Dataset Length

```python
def __len__(self):
    return len(self.input_ids)
```

Returns the total number of sequences.

Example

```
dataset size = 1000 sequences
```

---

#### 🧱 Step 11 — Getting One Sample

```python
def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]
```

Returns:

```
(input_sequence , target_sequence)
```

Example

```
Input  : tensor([10,11,12,13])
Target : tensor([11,12,13,14])
```

---

#### 🧱 Step 12 — DataLoader Creation

```python
def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True):
```

This function creates a **PyTorch DataLoader**.

---

### Step 1 — Load Tokenizer

```python
tokenizer = tiktoken.get_encoding("gpt2")
```

Uses GPT-2 tokenizer.

Vocabulary size:

```
50257 tokens
```

---

### Step 2 — Create Dataset

```python
dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
```

Creates dataset object.

---

### Step 3 — Create DataLoader

```python
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
```

DataLoader will:

* group sequences into batches
* shuffle dataset
* load efficiently

Example batch:

```
Batch size = 8
```

Output shape

```
Input  : (8 , 4)
Target : (8 , 4)
```

---

#### 🧱 Step 13 — Reading Training Data

```python
with open("/Users/anand/Desktop/LLM From Scratch/Dataset/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
```

Reads the text dataset.

Example:

```
Sherlock Holmes stories
novels
articles
```

---

#### 🧱 Step 14 — Encoding the Entire Text

```python
tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)
```

This converts the whole dataset to tokens.

Example

```
Text → Tokens
```

---

#### 🧱 Step 15 — Model Hyperparameters

```python
vocab_size = 50257
```

GPT-2 vocabulary size.

---

```python
output_dim = 256
```

Embedding dimension.

Each token becomes a **256-dimensional vector**.

---

```python
max_len = 1024
context_length = max_len
```

Maximum sequence length the model can process.

Example:

```
GPT2 = 1024 tokens
GPT3 = 2048 tokens
GPT4 = larger
```

---

#### 🧱 Step 16 — Token Embedding Layer

```python
token_embedding_layer = nn.Embedding(vocab_size, output_dim)
```

Transforms tokens into vectors.

Shape:

```
(50257 , 256)
```

Example

```
Token 100 → 256 dimensional vector
```

---

### Visualization

```
Token ID
   │
   ▼
Embedding Table
   │
   ▼
Vector (256)
```

---

#### 🧱 Step 17 — Positional Embedding

```python
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
```

Transformers **do not understand order**, so positional embeddings add sequence position.

Example

```
Position 0 → vector
Position 1 → vector
Position 2 → vector
```

Shape:

```
(1024 , 256)
```

---

#### 🧱 Step 18 — Creating the DataLoader

```python
max_length = 4
```

Small sequence length for testing.

---

```python
dataloader = create_dataloader(
    raw_text,
    batch_size=8,
    max_length=max_length,
    stride=max_length
)
```

Parameters:

| Parameter       | Value |
| --------------- | ----- |
| batch_size      | 8     |
| sequence length | 4     |
| stride          | 4     |

---

#### 🧠 Example Batch

Example tokens:

```
[10,11,12,13,14,15,16,17]
```

Batch Input

```
[[10,11,12,13],
 [14,15,16,17],
 ...]
```

Batch Target

```
[[11,12,13,14],
 [15,16,17,18],
 ...]
```

Shape:

```
Input  : (8 , 4)
Target : (8 , 4)
```

---

#### 🔄 Full Pipeline

```
Raw Text
   │
   ▼
Tokenizer
   │
   ▼
Token IDs
   │
   ▼
Sliding Window
   │
   ▼
Input + Target sequences
   │
   ▼
Dataset
   │
   ▼
DataLoader
   │
   ▼
Batch of Tokens
   │
   ▼
Token Embedding
   │
   ▼
Positional Embedding
   │
   ▼
Transformer Model
```

---

#### 🎯 Key Idea

GPT training is simply:

```
Predict next token
```

Example:

```
Input  : "The cat sat"
Target : "cat sat on"
```

Loss function:

```
CrossEntropyLoss
```

---

✅ This pipeline prepares **text data for training a GPT model**.

---

In [1]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

    return dataloader


with open("/Users/anand/Desktop/LLM From Scratch/Dataset/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

vocab_size = 50257
output_dim = 256
max_len = 1024
context_length = max_len


token_embedding_layer = nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

max_length = 4
dataloader = create_dataloader(raw_text, batch_size=8, max_length=max_length, stride=max_length)

In [4]:
for batch in dataloader:
    x, y = batch

    token_embeddings = token_embedding_layer(x)
    pos_embeddings = pos_embedding_layer(torch.arange(max_length))

    input_embeddings = token_embeddings + pos_embeddings

    break

## Causal Multi head attention One of the Approach

This class implements **masked self-attention (causal attention)** used in GPT models.

The key idea:

```
A token can only attend to previous tokens
NOT future tokens
```

Example sequence:

```
Token1 Token2 Token3 Token4
```

Attention allowed:

```
Token1 → Token1
Token2 → Token1 Token2
Token3 → Token1 Token2 Token3
Token4 → Token1 Token2 Token3 Token4
```

Future tokens are **masked**.

---

## class definition

```python
class CausalSelfAttention(nn.Module):
```

This creates a **PyTorch module** for causal attention.

---

## initialization

```python
def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
```

### parameters

| parameter        | meaning                            |
| ---------------- | ---------------------------------- |
| `d_in`           | input embedding size               |
| `d_out`          | output dimension                   |
| `context_length` | maximum sequence length            |
| `dropout`        | dropout probability                |
| `qkv_bias`       | whether QKV linear layers use bias |

Example:

```
d_in = 256
d_out = 256
context_length = 1024
```

---

## query key value projections

```python
self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
```

These layers create **Query, Key, Value** vectors.

### transformation

```
input embedding
      ↓
linear layer
      ↓
Q , K , V
```

Example shapes

```
Input: (B, T, d_in)

Queries: (B, T, d_out)
Keys   : (B, T, d_out)
Values : (B, T, d_out)
```

Where:

```
B = batch size
T = number of tokens
```

---

## dropout

```python
self.dropout = nn.Dropout(dropout)
```

Dropout prevents **overfitting**.

It randomly zeroes some attention weights during training.

---

## causal mask

```python
self.register_buffer(
    'mask',
    torch.triu(torch.ones(context_length, context_length), diagonal=1)
)
```

This creates the **causal mask**.

### example mask

For sequence length = 4

```
0 1 1 1
0 0 1 1
0 0 0 1
0 0 0 0
```

Where:

```
1 = blocked attention
0 = allowed attention
```

`register_buffer` means:

```
not a trainable parameter
but stored with the model
```

---

## forward pass

```python
def forward(self, x):
```

Input shape:

```
x : (B, T, d_in)
```

Example:

```
B = 8
T = 4
d_in = 256
```

---

## extracting dimensions

```python
b, n_tokens, d_in = x.shape
```

Where

```
b = batch size
n_tokens = sequence length
d_in = embedding dimension
```

---

## computing q k v

```python
keys = self.W_key(x)
queries = self.W_query(x)
values = self.W_value(x)
```

Shapes:

```
keys    : (B, T, d_out)
queries : (B, T, d_out)
values  : (B, T, d_out)
```

---

## attention scores

```python
attn_scores = queries @ keys.transpose(1, 2)
```

Matrix multiplication:

```
Q × K^T
```

Shapes:

```
queries      : (B, T, d_out)
keys^T       : (B, d_out, T)

result       : (B, T, T)
```

Meaning:

```
each token compared with every other token
```

Example matrix:

```
Token1 Token2 Token3 Token4
```

```
[1.2 0.3 0.1 0.0]
[0.5 1.4 0.2 0.1]
[0.4 0.7 1.1 0.3]
[0.3 0.2 0.6 1.0]
```

---

## applying causal mask

```python
attn_scores.masked_fill_(
    self.mask.bool()[:n_tokens, :n_tokens],
    -torch.inf
)
```

Future tokens get **−∞ score**.

Example

Before mask

```
1.2 0.3 0.1 0.7
0.5 1.4 0.2 0.8
```

After mask

```
1.2 -inf -inf -inf
0.5 1.4 -inf -inf
```

---

## scaling attention

```python
attn_scores / keys.shape[-1]**0.5
```

This prevents **large dot product values**.

Formula:

```
Attention(Q,K,V) =
softmax(QKᵀ / √d) V
```

Where:

```
d = key dimension
```

---

## softmax

```python
attn_weights = torch.softmax(... , dim=-1)
```

Softmax converts scores → probabilities.

Example

```
[2.0 1.0 0.5]
```

becomes

```
[0.62 0.23 0.15]
```

Sum = 1.

---

## dropout on attention

```python
attn_weights = self.dropout(attn_weights)
```

Randomly removes some attention connections during training.

---

## computing context vector

```python
context_vec = attn_weights @ values
```

Shapes:

```
attn_weights : (B, T, T)
values       : (B, T, d_out)

result       : (B, T, d_out)
```

Meaning:

```
weighted combination of value vectors
```

Output:

```
context_vec
```

This is the **attention output representation**.

---

# multi head attention wrapper

This class creates **multiple attention heads**.

Each head learns **different relationships**.

---

## class definition

```python
class MultiHeadAttentionWrapper(nn.Module):
```

---

## initialization

```python
def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
```

### parameters

| parameter   | meaning                   |
| ----------- | ------------------------- |
| `num_heads` | number of attention heads |
| `d_in`      | input dimension           |
| `d_out`     | head dimension            |

Example

```
num_heads = 8
d_out = 64
```

---

## creating multiple heads

```python
self.heads = nn.ModuleList(
    [CausalSelfAttention(...) for _ in range(num_heads)]
)
```

Example:

```
head1
head2
head3
...
head8
```

Each head processes the **same input** but learns different patterns.

---

## output projection

```python
self.out_proj = nn.Linear(d_out*num_heads, d_out*num_heads)
```

After concatenation we project back into a unified representation.

---

## forward pass

```python
def forward(self, x):
```

Input

```
x : (B, T, d_in)
```

---

## computing each head

```python
[head(x) for head in self.heads]
```

Each head produces:

```
(B, T, d_out)
```

Example

```
Head1 → (B,T,64)
Head2 → (B,T,64)
Head3 → (B,T,64)
Head4 → (B,T,64)
```

---

## concatenation

```python
torch.cat(... , dim=-1)
```

Concatenation dimension = embedding dimension.

Example

```
(B,T,64)
(B,T,64)
(B,T,64)
(B,T,64)
```

becomes

```
(B,T,256)
```

---

## final projection

```python
return self.out_proj(context_vec)
```

Linear layer mixes information from all heads.

Final output shape:

```
(B , T , num_heads * d_out)
```

---

# full multi head attention flow

```
input embeddings
      │
      ▼
split into multiple heads
      │
      ▼
head1 attention
head2 attention
head3 attention
...
      │
      ▼
concatenate outputs
      │
      ▼
linear projection
      │
      ▼
final attention output
```

---

✅ This implementation is **simple educational MHA**.

Modern models optimize it by:

* fused QKV projection
* flash attention
* grouped query attention
* rotary embeddings
* KV cache

---


In [2]:
class CausalSelfAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # New
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, n_tokens, d_in = x.shape # New batch dimension b
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:n_tokens, :n_tokens], -torch.inf) 
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec


class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalSelfAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )
        self.out_proj = nn.Linear(d_out*num_heads, d_out*num_heads)

    def forward(self, x):
        context_vec = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.out_proj(context_vec)

In [5]:
torch.manual_seed(123)

context_length = max_length
d_in = output_dim

num_heads=2
d_out = d_in // num_heads

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads)

batch = input_embeddings
context_vecs = mha(batch)

print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([8, 4, 256])


## Multi head attention (optimized implementation)

This class implements **Multi-Head Self Attention (MHA)** used in Transformer models like GPT.

Key idea:

```text
Instead of one attention operation,
we run multiple attention heads in parallel.
```

Each head learns **different relationships between tokens**.

Example:

| Head   | What it might learn     |
| ------ | ----------------------- |
| Head 1 | grammar                 |
| Head 2 | long-range dependencies |
| Head 3 | semantic meaning        |
| Head 4 | positional patterns     |

---

# class definition

```python
class MultiHeadAttention(nn.Module):
```

This is a **PyTorch neural network module** implementing multi-head attention.

---

# initialization

```python
def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
```

### parameters

| parameter        | meaning                    |
| ---------------- | -------------------------- |
| `d_in`           | input embedding dimension  |
| `d_out`          | output embedding dimension |
| `context_length` | maximum sequence length    |
| `dropout`        | dropout probability        |
| `num_heads`      | number of attention heads  |
| `qkv_bias`       | bias in QKV projections    |

---

# ensuring valid head split

```python
assert d_out % num_heads == 0
```

The embedding dimension must split evenly across heads.

Example:

```text
d_out = 256
num_heads = 8
```

Each head dimension becomes:

```text
head_dim = 256 / 8 = 32
```

---

# storing dimensions

```python
self.d_out = d_out
self.num_heads = num_heads
self.head_dim = d_out // num_heads
```

So each head processes **`head_dim` features**.

---

# query key value projections

```python
self.W_query = nn.Linear(d_in, d_out)
self.W_key   = nn.Linear(d_in, d_out)
self.W_value = nn.Linear(d_in, d_out)
```

These transform input embeddings into:

```text
Queries
Keys
Values
```

Example shapes:

```text
Input x: (B, T, d_in)

Queries: (B, T, d_out)
Keys   : (B, T, d_out)
Values : (B, T, d_out)
```

Where:

```
B = batch size
T = number of tokens
```

---

# output projection

```python
self.out_proj = nn.Linear(d_out, d_out)
```

After combining heads, we mix their information using a linear layer.

---

# dropout

```python
self.dropout = nn.Dropout(dropout)
```

Prevents overfitting by randomly removing attention connections during training.

---

# causal mask

```python
self.register_buffer(
    'mask',
    torch.triu(torch.ones(context_length, context_length), diagonal=1)
)
```

This creates a **causal attention mask**.

Example mask (sequence length = 4):

```
0 1 1 1
0 0 1 1
0 0 0 1
0 0 0 0
```

Meaning:

```
1 → block attention
0 → allow attention
```

This ensures **tokens cannot see future tokens**.

---

# forward pass

```python
def forward(self, x):
```

Input shape:

```
x = (B, T, d_in)
```

Example:

```
B = 8
T = 4
d_in = 256
```

---

# extracting dimensions

```python
b, num_tokens, d_in = x.shape
```

Where:

```
b = batch size
num_tokens = sequence length
d_in = embedding dimension
```

---

# computing q k v

```python
keys = self.W_key(x)
queries = self.W_query(x)
values = self.W_value(x)
```

Shape:

```
(B, T, d_out)
```

---

# splitting into multiple heads

```python
keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
```

Before split:

```
(B, T, d_out)
```

After split:

```
(B, T, num_heads, head_dim)
```

Example:

```
(8, 4, 256)
↓
(8, 4, 8, 32)
```

Same operation is applied to:

```
queries
values
```

---

# rearranging dimensions

```python
keys = keys.transpose(1, 2)
queries = queries.transpose(1, 2)
values = values.transpose(1, 2)
```

Shape becomes:

```
(B, num_heads, T, head_dim)
```

Example:

```
(8, 4, 8, 32)
↓
(8, 8, 4, 32)
```

This allows attention to run **independently per head**.

---

# attention score computation

```python
attn_scores = queries @ keys.transpose(2, 3)
```

Matrix multiplication:

```
Q × Kᵀ
```

Shapes:

```
queries  : (B, H, T, D)
keys^T   : (B, H, D, T)

result   : (B, H, T, T)
```

Meaning:

```
Each token compares with every other token
inside each head.
```

---

# applying causal mask

```python
mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
attn_scores.masked_fill_(mask_bool, -torch.inf)
```

Future tokens get **−∞ score**.

Example:

Before mask

```
[1.2 0.8 0.5 0.7]
```

After mask

```
[1.2 -inf -inf -inf]
```

This prevents **future attention**.

---

# scaling attention scores

```python
attn_scores / keys.shape[-1]**0.5
```

This avoids extremely large values.

Formula:

```
Attention(Q,K,V) =
softmax(QKᵀ / √d_k) V
```

Where:

```
d_k = head_dim
```

---

# softmax

```python
attn_weights = torch.softmax(..., dim=-1)
```

Converts scores into probabilities.

Example:

```
[2.0 1.0 0.5]
```

becomes

```
[0.62 0.23 0.15]
```

Sum = 1.

---

# dropout on attention

```python
attn_weights = self.dropout(attn_weights)
```

Randomly removes some attention weights during training.

---

# computing context vectors

```python
context_vec = (attn_weights @ values)
```

Shapes:

```
attn_weights : (B, H, T, T)
values       : (B, H, T, D)

result       : (B, H, T, D)
```

Each token becomes a **weighted combination of value vectors**.

---

# restoring token order

```python
context_vec = context_vec.transpose(1, 2)
```

Shape becomes:

```
(B, T, H, D)
```

Example:

```
(8, 8, 4, 32)
↓
(8, 4, 8, 32)
```

---

# merging heads

```python
context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
```

Before merge:

```
(B, T, H, D)
```

After merge:

```
(B, T, H × D)
```

Example:

```
(8,4,8,32)
↓
(8,4,256)
```

This reconstructs the original embedding dimension.

---

# final projection

```python
context_vec = self.out_proj(context_vec)
```

This mixes information from different heads.

Final output shape:

```
(B, T, d_out)
```

---

# full attention pipeline

```
Input embeddings
        │
        ▼
Linear projections
(Q, K, V)
        │
        ▼
Split into heads
        │
        ▼
Scaled dot-product attention
        │
        ▼
Causal masking
        │
        ▼
Softmax
        │
        ▼
Weighted value aggregation
        │
        ▼
Merge heads
        │
        ▼
Linear projection
        │
        ▼
Final attention output
```

---

# key differences vs the wrapper version

| Wrapper Version                | This Implementation              |
| ------------------------------ | -------------------------------- |
| Each head is a separate module | All heads computed in one tensor |
| Slower                         | Faster                           |
| Easier to understand           | More efficient                   |
| Used for teaching              | Used in real models              |

Modern LLMs always use **this vectorized version**.

---

✅ This implementation is very close to what models like:

* GPT-2
* GPT-3
* LLaMA
* Mistral

use internally.

---


In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head
        
        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [7]:
torch.manual_seed(123)

context_length = max_length
d_in = output_dim
d_out = d_in

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

batch = input_embeddings
context_vecs = mha(batch)

print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([8, 4, 256])


## Overall attention pipeline

![Image](https://uvadlc-notebooks.readthedocs.io/en/latest/_images/transformer_architecture.svg)

![Image](https://miro.medium.com/0%2A4q89aLITCfW5qm6W.png)

![Image](https://miro.medium.com/v2/resize%3Afit%3A960/1%2ANMneDGsvnXyOFqN6m8uSyA.png)

![Image](https://uvadlc-notebooks.readthedocs.io/en/latest/_images/tutorial_notebooks_tutorial6_Transformers_and_MHAttention_90_2.svg)

The flow inside your implementation is:

```
Input Embeddings
       │
       ▼
Linear Layers → Q K V
       │
       ▼
Split into multiple heads
       │
       ▼
Scaled Dot Product Attention (per head)
       │
       ▼
Concatenate heads
       │
       ▼
Linear Projection
       │
       ▼
Final Output
```

---

### step 1 — input tensor

Your input:

```python
x
```

Shape:

```
(B, T, d_in)
```

Example:

```
B = 8   (batch size)
T = 4   (tokens)
d_in = 256 (embedding size)
```

So:

```
x = (8,4,256)
```

Each token has a **256-dimensional vector**.

---

### step 2 — q k v projection

From the code:

```python
queries = self.W_query(x)
keys = self.W_key(x)
values = self.W_value(x)
```

Each projection:

```
(B, T, d_out)
```

Example:

```
(8,4,256)
```

Visual:

```
Input Embedding
      │
      ▼
 ┌───────────────┐
 │ Linear Layers │
 └───────────────┘
   │      │      │
   ▼      ▼      ▼
   Q      K      V
```

---

### step 3 — splitting into heads

Code:

```python
keys.view(b, num_tokens, num_heads, head_dim)
```

Suppose:

```
d_out = 256
num_heads = 8
```

Then:

```
head_dim = 256 / 8 = 32
```

Shape becomes:

```
(B, T, H, D)
```

Example:

```
(8,4,8,32)
```

Visual:

```
256 embedding
      │
      ▼
Split into heads
```

```
Head1 → 32
Head2 → 32
Head3 → 32
Head4 → 32
Head5 → 32
Head6 → 32
Head7 → 32
Head8 → 32
```

---

### step 4 — rearranging dimensions

Code:

```python
keys.transpose(1,2)
```

Shape changes:

```
(B,T,H,D)
      ↓
(B,H,T,D)
```

Example:

```
(8,4,8,32)
      ↓
(8,8,4,32)
```

Why?

Because attention is computed **per head**.

---

### step 5 — attention scores

Code:

```python
attn_scores = queries @ keys.transpose(2,3)
```

Shapes:

```
queries  (B,H,T,D)
keys^T   (B,H,D,T)
```

Result:

```
(B,H,T,T)
```

Example:

```
(8,8,4,4)
```

Meaning:

```
each token attends to every other token
inside each head
```

Matrix example (for one head):

```
Token1 Token2 Token3 Token4
```

```
[0.8 0.1 0.2 0.0]
[0.3 0.9 0.2 0.1]
[0.4 0.3 1.2 0.2]
[0.1 0.5 0.6 0.7]
```

---

### step 6 — causal masking

Your mask prevents future tokens.

Example:

```
Token1 Token2 Token3 Token4
```

Mask:

```
0 1 1 1
0 0 1 1
0 0 0 1
0 0 0 0
```

After masking:

```
Token1 → only Token1
Token2 → Token1 Token2
Token3 → Token1 Token2 Token3
Token4 → Token1 Token2 Token3 Token4
```

---

### step 7 — softmax

Code:

```python
attn_weights = softmax(...)
```

Example:

Before:

```
[2.1 1.3 0.4]
```

After:

```
[0.63 0.28 0.09]
```

Sum = 1.

---

### step 8 — weighted value vectors

Code:

```python
context = attn_weights @ values
```

Shapes:

```
(B,H,T,T)
×
(B,H,T,D)
```

Result:

```
(B,H,T,D)
```

Example:

```
(8,8,4,32)
```

Meaning:

```
Each token becomes a weighted sum of value vectors.
```

---

### step 9 — merging heads

Code:

```python
transpose(1,2)
view(...)
```

Shape transformation:

```
(B,H,T,D)
     ↓
(B,T,H,D)
     ↓
(B,T,H*D)
```

Example:

```
(8,8,4,32)
     ↓
(8,4,8,32)
     ↓
(8,4,256)
```

This restores the **original embedding dimension**.

---

### step 10 — output projection

Code:

```python
self.out_proj(context_vec)
```

Shape stays:

```
(B,T,256)
```

But the linear layer **mixes information from different heads**.

---

### final tensor flow

```
Input (B,T,256)
        │
        ▼
Q,K,V projection
        │
        ▼
(B,T,256)
        │
        ▼
Split heads
        │
        ▼
(B,H,T,32)
        │
        ▼
Attention scores
        │
        ▼
(B,H,T,T)
        │
        ▼
Softmax
        │
        ▼
Weighted values
        │
        ▼
(B,H,T,32)
        │
        ▼
Merge heads
        │
        ▼
(B,T,256)
        │
        ▼
Linear projection
        │
        ▼
Final Output
```

---

💡 **Important intuition**

Multi-Head Attention allows the model to look at **different relationships simultaneously**.

Example sentence:

```
"The animal didn't cross the street because it was tired"
```

Different heads may learn:

| head  | learns             |
| ----- | ------------------ |
| head1 | grammar            |
| head2 | long dependencies  |
| head3 | pronoun resolution |
| head4 | semantic meaning   |

---